# Query 5: 7-Day Moving Average of Fare
**Type:** Window Function (Moving Average)  
**Problem Statement:** Calculate a 7-day sliding window moving average of fare amounts to smooth daily fare trends. This reveals seasonal patterns and removes daily noise from the data.

In [1]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, IntegerType

spark = SparkSession.builder \
    .appName('Q5_MovingAverage') \
    .master('local[*]') \
    .config('spark.sql.shuffle.partitions', '8') \
    .config('spark.driver.memory', '2g') \
    .config('spark.executor.memory', '2g') \
    .config('spark.eventLog.enabled', 'false') \
    .config('spark.ui.enabled', 'false') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

26/04/25 20:17:13 WARN Utils: Your hostname, mariam-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/04/25 20:17:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/25 20:17:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.1


In [2]:
df = spark.read.option('header', 'true').option('inferSchema', 'true') \
          .csv('../data/yellow_tripdata_2015-01.csv') \
          .sample(fraction=0.2, seed=42)

df = df.withColumnRenamed('tpep_pickup_datetime',  'pickup_datetime') \
       .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime') \
       .withColumnRenamed('fare_amount',            'fare') \
       .withColumnRenamed('passenger_count',        'passengers') \
       .withColumnRenamed('trip_distance',          'distance') \
       .withColumnRenamed('total_amount',           'total')

df = df.withColumn('fare',       F.col('fare').cast(DoubleType())) \
       .withColumn('total',      F.col('total').cast(DoubleType())) \
       .withColumn('distance',   F.col('distance').cast(DoubleType())) \
       .withColumn('passengers', F.col('passengers').cast(IntegerType())) \
       .cache()

df.createOrReplaceTempView('trips')
rdd = df.rdd
print('Total rows (20% sample):', df.count())

Total rows (20% sample): 2233826


## RDD Implementation

In [3]:
start = time.time()

# RDD cannot do distributed window functions.
# Workaround: compute daily avg on cluster, then apply sliding window on driver.
daily = (
    rdd
    .filter(lambda r: r['pickup_datetime'] is not None
                  and r['fare']            is not None)
    .map(lambda r: (str(r['pickup_datetime'].date()), (r['fare'], 1)))
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
    .mapValues(lambda x: x[0] / x[1])
    .sortBy(lambda x: x[0])
    .collect()
)

# Apply 7-day sliding window on the driver (not distributed)
result_rdd = []
for i, (day, avg) in enumerate(daily):
    start_i = max(0, i - 6)
    mov_avg = sum(v for _, v in daily[start_i:i+1]) / (i - start_i + 1)
    result_rdd.append((day, round(avg, 2), round(mov_avg, 2)))

rdd_time = time.time() - start
print(f'RDD | Days computed: {len(result_rdd)} | Time: {rdd_time:.2f}s')
print('7-day moving avg – first 10 days (RDD):')
for day, avg, mov in result_rdd[:10]:
    print(f'  {day}  daily_avg=${avg}  7day_mov_avg=${mov}')

RDD | Days computed: 31 | Time: 22.32s
7-day moving avg – first 10 days (RDD):
  2015-01-01  daily_avg=$12.66  7day_mov_avg=$12.66
  2015-01-02  daily_avg=$12.13  7day_mov_avg=$12.4
  2015-01-03  daily_avg=$11.63  7day_mov_avg=$12.14
  2015-01-04  daily_avg=$12.76  7day_mov_avg=$12.3
  2015-01-05  daily_avg=$12.15  7day_mov_avg=$12.27
  2015-01-06  daily_avg=$11.83  7day_mov_avg=$12.19
  2015-01-07  daily_avg=$11.46  7day_mov_avg=$12.09
  2015-01-08  daily_avg=$11.69  7day_mov_avg=$11.95
  2015-01-09  daily_avg=$11.98  7day_mov_avg=$11.93
  2015-01-10  daily_avg=$11.33  7day_mov_avg=$11.88


## DataFrame Implementation

In [4]:
start = time.time()

w = Window.orderBy('pickup_datetime').rowsBetween(-6, 0)

result_df = (
    df.filter(F.col('fare').isNotNull())
      .withColumn('trip_date', F.to_date('pickup_datetime'))
      .select('pickup_datetime', 'trip_date', 'fare')
      .withColumn('moving_avg_7',
                  F.round(F.avg('fare').over(w), 2))
      .orderBy('pickup_datetime')
)

print('--- Q5 DataFrame explain(True) ---')
result_df.explain(True)
result_df.cache()
df_time = time.time() - start
print(f'DataFrame | Time: {df_time:.2f}s')
result_df.show(20)
result_df.unpersist()

--- Q5 DataFrame explain(True) ---
== Parsed Logical Plan ==
'Sort ['pickup_datetime ASC NULLS FIRST], true
+- Project [pickup_datetime#55, trip_date#1249, fare#176, moving_avg_7#1274]
   +- Project [pickup_datetime#55, trip_date#1249, fare#176, _we0#1275, round(_we0#1275, 2) AS moving_avg_7#1274]
      +- Window [avg(fare#176) windowspecdefinition(pickup_datetime#55 ASC NULLS FIRST, specifiedwindowframe(RowFrame, -6, currentrow$())) AS _we0#1275], [pickup_datetime#55 ASC NULLS FIRST]
         +- Project [pickup_datetime#55, trip_date#1249, fare#176]
            +- Project [pickup_datetime#55, trip_date#1249, fare#176]
               +- Project [VendorID#17, pickup_datetime#55, dropoff_datetime#76, passengers#236, distance#216, pickup_longitude#22, pickup_latitude#23, RateCodeID#24, store_and_fwd_flag#25, dropoff_longitude#26, dropoff_latitude#27, payment_type#28, fare#176, extra#30, mta_tax#31, tip_amount#32, tolls_amount#33, improvement_surcharge#34, total#196, to_date(pickup_datetim

+-------------------+----------+----+------------+
|    pickup_datetime| trip_date|fare|moving_avg_7|
+-------------------+----------+----+------------+
|2015-01-01 00:00:05|2015-01-01| 8.5|         8.5|
|2015-01-01 00:00:08|2015-01-01| 9.5|         9.0|
|2015-01-01 00:00:09|2015-01-01|13.0|       10.33|
|2015-01-01 00:00:09|2015-01-01| 9.5|       10.13|
|2015-01-01 00:00:11|2015-01-01| 9.5|        10.0|
|2015-01-01 00:00:11|2015-01-01| 5.0|        9.17|
|2015-01-01 00:00:25|2015-01-01|13.0|        9.71|
|2015-01-01 00:00:26|2015-01-01| 8.5|        9.71|
|2015-01-01 00:00:27|2015-01-01| 9.0|        9.64|
|2015-01-01 00:00:28|2015-01-01| 6.0|        8.64|
|2015-01-01 00:00:30|2015-01-01| 5.5|        8.07|
|2015-01-01 00:00:32|2015-01-01| 6.5|        7.64|
|2015-01-01 00:00:38|2015-01-01|32.0|        11.5|
|2015-01-01 00:00:41|2015-01-01|44.0|       15.93|
|2015-01-01 00:00:42|2015-01-01|19.0|       17.43|
|2015-01-01 00:00:48|2015-01-01|10.5|       17.64|
|2015-01-01 00:00:50|2015-01-01

DataFrame[pickup_datetime: timestamp, trip_date: date, fare: double, moving_avg_7: double]

## Spark SQL Implementation

In [5]:
start = time.time()

result_sql = spark.sql("""
    SELECT   pickup_datetime,
             DATE(pickup_datetime) AS trip_date,
             fare,
             ROUND(AVG(fare) OVER (
                 ORDER BY pickup_datetime
                 ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
             ), 2) AS moving_avg_7
    FROM     trips
    WHERE    fare IS NOT NULL
    ORDER BY pickup_datetime
""")

print('--- Q5 Spark SQL explain(True) ---')
result_sql.explain(True)
result_sql.cache()
sql_time = time.time() - start
print(f'SQL | Time: {sql_time:.2f}s')
result_sql.show(20)
result_sql.unpersist()

--- Q5 Spark SQL explain(True) ---
== Parsed Logical Plan ==
'Sort ['pickup_datetime ASC NULLS FIRST], true
+- 'Project ['pickup_datetime, 'DATE('pickup_datetime) AS trip_date#2159, 'fare, 'ROUND('AVG('fare) windowspecdefinition('pickup_datetime ASC NULLS FIRST, specifiedwindowframe(RowFrame, -6, currentrow$())), 2) AS moving_avg_7#2160]
   +- 'Filter isnotnull('fare)
      +- 'UnresolvedRelation [trips], [], false

== Analyzed Logical Plan ==
pickup_datetime: timestamp, trip_date: date, fare: double, moving_avg_7: double
Sort [pickup_datetime#55 ASC NULLS FIRST], true
+- Project [pickup_datetime#55, trip_date#2159, fare#176, moving_avg_7#2160]
   +- Project [pickup_datetime#55, trip_date#2159, fare#176, _we0#2162, round(_we0#2162, 2) AS moving_avg_7#2160]
      +- Window [avg(fare#176) windowspecdefinition(pickup_datetime#55 ASC NULLS FIRST, specifiedwindowframe(RowFrame, -6, currentrow$())) AS _we0#2162], [pickup_datetime#55 ASC NULLS FIRST]
         +- Project [pickup_datetime#55, c

+-------------------+----------+----+------------+
|    pickup_datetime| trip_date|fare|moving_avg_7|
+-------------------+----------+----+------------+
|2015-01-01 00:00:05|2015-01-01| 8.5|         8.5|
|2015-01-01 00:00:08|2015-01-01| 9.5|         9.0|
|2015-01-01 00:00:09|2015-01-01|13.0|       10.33|
|2015-01-01 00:00:09|2015-01-01| 9.5|       10.13|
|2015-01-01 00:00:11|2015-01-01| 9.5|        10.0|
|2015-01-01 00:00:11|2015-01-01| 5.0|        9.17|
|2015-01-01 00:00:25|2015-01-01|13.0|        9.71|
|2015-01-01 00:00:26|2015-01-01| 8.5|        9.71|
|2015-01-01 00:00:27|2015-01-01| 9.0|        9.64|
|2015-01-01 00:00:28|2015-01-01| 6.0|        8.64|
|2015-01-01 00:00:30|2015-01-01| 5.5|        8.07|
|2015-01-01 00:00:32|2015-01-01| 6.5|        7.64|
|2015-01-01 00:00:38|2015-01-01|32.0|        11.5|
|2015-01-01 00:00:41|2015-01-01|44.0|       15.93|
|2015-01-01 00:00:42|2015-01-01|19.0|       17.43|
|2015-01-01 00:00:48|2015-01-01|10.5|       17.64|
|2015-01-01 00:00:50|2015-01-01

DataFrame[pickup_datetime: timestamp, trip_date: date, fare: double, moving_avg_7: double]

## Performance Comparison

In [6]:
print('='*65)
row1 = f'{"Metric":<25} {"RDD":>12} {"DataFrame":>12} {"SQL":>12}'
row2 = f'{"Execution Time":<25} {rdd_time:>11.2f}s {df_time:>11.2f}s {sql_time:>11.2f}s'
row3 = f'{"Window Function":<25} {"Manual":>12} {"Built-in":>12} {"Built-in":>12}'
row4 = f'{"Distributed":<25} {"Partial":>12} {"Yes":>12} {"Yes":>12}'
row5 = f'{"Optimizer":<25} {"None":>12} {"Catalyst":>12} {"Catalyst":>12}'
print(row1)
print('-'*65)
print(row2)
print(row3)
print(row4)
print(row5)
print('='*65)
print()
print('KEY INSIGHT:')
print('RDD computes daily averages on cluster but sliding window on driver.')
print('DataFrame/SQL use Tungsten Window operator — fully distributed.')
print('ROWS BETWEEN 6 PRECEDING AND CURRENT ROW is native in Spark SQL.')

Metric                             RDD    DataFrame          SQL
-----------------------------------------------------------------
Execution Time                  22.32s        0.34s        0.46s
Window Function                 Manual     Built-in     Built-in
Distributed                    Partial          Yes          Yes
Optimizer                         None     Catalyst     Catalyst

KEY INSIGHT:
RDD computes daily averages on cluster but sliding window on driver.
DataFrame/SQL use Tungsten Window operator — fully distributed.
ROWS BETWEEN 6 PRECEDING AND CURRENT ROW is native in Spark SQL.
